# Part 1 - Multimodal RAG
DATA 266 Lab 2, Pair_Programming_Team_32. The code lives in `part1_rag/src/`; this notebook runs it.
Run on Colab (CPU is enough): Runtime → Restart & Run All.

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
except ImportError:
    gpu = "cpu"
print({"commit": commit, "python": sys.version.split()[0], "gpu": gpu, "seed": SEED})

## Setup
Colab secrets needed: `GITHUB_TOKEN` (the repo is private) and `GEMINI_API_KEY`.

In [ ]:
import os
BRANCH = "zoheb"                                            # switch to main once merged
if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    if not os.path.isdir("data266-lab2-team32"):
        !git clone -q -b {BRANCH} https://{userdata.get("GITHUB_TOKEN")}@github.com/sakotiya/data266-lab2-team32.git
    %cd data266-lab2-team32
    !pip install -q pymupdf chromadb sentence-transformers rank-bm25 google-genai pydantic
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import chromadb, pymupdf, sentence_transformers, google.genai
from part1_rag.src import common
print({"commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
       "pymupdf": pymupdf.__version__, "chromadb": chromadb.__version__,
       "sentence-transformers": sentence_transformers.__version__, "google-genai": google.genai.__version__,
       "model": common.GEMINI_MODEL})

## Data
The competition files are not in git. Upload the 3 PDFs and `Questions.csv` from Kaggle when asked.

In [ ]:
need = [d["file"] for d in common.DOCS.values()] + ["Questions.csv"]
missing = [f for f in need if not (common.DATA / f).exists()]
if missing and "google.colab" in sys.modules:
    from google.colab import files
    print("upload:", missing)
    for name, data in files.upload().items():
        (common.DATA / name.replace(" (1)", "")).write_bytes(data)
missing = [f for f in need if not (common.DATA / f).exists()]
assert not missing, f"missing data files: {missing}"

## 1.1 Ingest and index
Pages become text chunks; every figure and table is cropped. Figure transcriptions come from the committed
cache `outputs/figure_descriptions.json`, so this step makes no API calls.

In [ ]:
from part1_rag.src import ingest, index
chunks, figures = ingest.ingest_all()
index.build_index()

## 1.2-1.5 Answer Questions.csv
One Gemini call per question (40 for the public set). Free tier: about 20 requests/day on Flash models, more on flash-lite.

In [ ]:
from part1_rag.src import pipeline
sub, seconds = pipeline.run()
print("median seconds/question:", round(float(np.median(list(seconds.values()))), 2))
print(sub["status"].value_counts().to_dict())
assert pipeline.validate()
sub.head(10)

In [ ]:
if "google.colab" in sys.modules:
    files.download(str(common.OUTPUTS / "submission.csv"))